# Statistical Analysis & Non‑LLM Machine Learning Workshop (Practice Notebook)


This notebook matches the Netlify website practice sections:

- **1.7 Hands‑On Statistics Practice (6:30–7:00)**
- **2.7 Hands‑On ML Practice (7:30–8:00)**

> Tip: If you want to look back at the instructor demo code, open the **Demo notebook**.

---

## 0. Setup (run once)

If you're on Colab, you can run this cell to make sure you have the needed packages.

In [ ]:
!pip -q install pandas numpy matplotlib scipy statsmodels scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["figure.dpi"] = 120


## 0.1 Load your dataset

You can use the <strong>Palmer Penguins</strong> dataset (<code>Penguindata.csv</code>) **or** your own hackathon dataset.

### Option A: load from a URL
Set `DATA_URL` to your hosted CSV (Netlify or GitHub raw).

### Option B: upload manually
Leave `DATA_URL` blank and upload a file when prompted.

In [ ]:
DATA_URL = ""  # e.g., "https://YOUR-SITE.netlify.app/Penguindata.csv"

LOCAL_CANDIDATES = [
    "/content/Penguindata.csv",
    "/content/data/Penguindata.csv",
]

def load_data():
    if DATA_URL.strip():
        print(f"Loading from URL: {DATA_URL}")
        return pd.read_csv(DATA_URL)

    for p in LOCAL_CANDIDATES:
        if Path(p).exists():
            print(f"Loading from local path: {p}")
            return pd.read_csv(p)

    try:
        from google.colab import files  # type: ignore
        print("Dataset not found. Please upload your CSV file now...")
        uploaded = files.upload()
        filename = next(iter(uploaded))
        print(f"Loaded uploaded file: {filename}")
        return pd.read_csv(filename)
    except Exception as e:
        raise FileNotFoundError(
            "Could not find a dataset locally and DATA_URL is empty. "
            "Set DATA_URL OR upload the file in Colab."
        ) from e

df = load_data()

# Small cleanup: strip trailing spaces in column names (this dataset has one like 'Latitude ')
df.columns = df.columns.str.strip()

print("Rows, Columns:", df.shape)
df.head()

### 0.2 Column helper


In [ ]:
def detect_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

COL_SPECIES = detect_col(df, ["Species", "species"])
COL_ISLAND = detect_col(df, ["Island", "island"])
COL_SEX = detect_col(df, ["Sex", "sex"])

COL_CULMEN_LEN = detect_col(df, ["Culmen Length (mm)", "Bill Length (mm)", "Culmen Length"])
COL_CULMEN_DEPTH = detect_col(df, ["Culmen Depth (mm)", "Bill Depth (mm)", "Culmen Depth"])
COL_FLIPPER = detect_col(df, ["Flipper Length (mm)", "Flipper Length"])
COL_BODY_MASS = detect_col(df, ["Body Mass (g)", "Body Mass"])

print("Detected columns (may be None):")
print("  COL_SPECIES     =", COL_SPECIES)
print("  COL_ISLAND      =", COL_ISLAND)
print("  COL_SEX         =", COL_SEX)
print("  COL_CULMEN_LEN  =", COL_CULMEN_LEN)
print("  COL_CULMEN_DEPTH=", COL_CULMEN_DEPTH)
print("  COL_FLIPPER     =", COL_FLIPPER)
print("  COL_BODY_MASS   =", COL_BODY_MASS)

# If needed, override here:
# COL_SPECIES = "..."
# COL_ISLAND = "..."
# COL_SEX = "..."
# COL_CULMEN_LEN = "..."
# COL_CULMEN_DEPTH = "..."
# COL_FLIPPER = "..."
# COL_BODY_MASS = "..."

# 1.7 Hands‑On Statistics Practice

Pick a question, choose a test, run it, and interpret it.

**Reminder:** You do NOT need to complete every test below.  
For the practice session, choose at least **one** test path (t-test OR chi-square OR ANOVA).

---

## 1.7A Beginner — Descriptive statistics

**Your task**
1. Choose one numeric column (You can try <code>Flipper Length (mm)</code> this time).  
2. Compute mean, median, variance (and/or `.describe()`).  
3. Plot a histogram and describe the distribution in 1–2 sentences.

In [ ]:
# TODO: choose a numeric column from your dataset
# Hint: COL_FLIPPER if you are going for flipper length


## 1.7B Medium — Run ONE hypothesis test

### Choose ONE path:
- **Path 1:** t-test (numeric outcome, 2 groups)  
- **Path 2:** chi-square (categorical × categorical)  
- **Path 3:** ANOVA (numeric outcome, 3+ groups)

**Your task**
- Write your hypothesis (H₀ and H₁) in plain English.
- Run the test.
- Interpret the p-value in plain English (no “cause” language).

> Tip: If you’re unsure which test to use, ask: “Is my outcome numeric or categorical?”

### 1.7B‑1 t-test template (numeric outcome, 2 groups)

Example idea (penguins): compare mean <code>Flipper length (g)</code> between two species (e.g., Adelie vs Gentoo), or between sexes.

Fill in:
- `OUTCOME_COL` (numeric)
- `GROUP_COL` (categorical)
- `GROUP_A` and `GROUP_B` (two categories)

In [ ]:
# TODO: Fill these in

# OUTCOME_COL should be a NUMERIC measurement.
# Hint: pick something measured in grams or millimeters.
# Ask yourself: "What average am I comparing?"
OUTCOME_COL = None

# GROUP_COL should be a CATEGORICAL column with exactly TWO groups.
# Hint: examples of categorical variables are species, sex, island.
# Make sure the column you choose actually contains only two categories
# for GROUP_A and GROUP_B.
GROUP_COL = None

# GROUP_A and GROUP_B must be TWO category names
# that appear exactly as written in the dataset.
# Hint: Use df[GROUP_COL].unique() to see the exact spelling.
GROUP_A = None
GROUP_B = None

# Basic checks (do not change)
assert OUTCOME_COL is not None and GROUP_COL is not None, "Set OUTCOME_COL and GROUP_COL."
assert GROUP_A is not None and GROUP_B is not None, "Set GROUP_A and GROUP_B to two group names."

# Extract the two groups
# (We convert to numeric just in case there are missing or messy values.)
g1 = pd.to_numeric(
    df.loc[df[GROUP_COL].astype(str) == str(GROUP_A), OUTCOME_COL],
    errors="coerce"
).dropna()

g2 = pd.to_numeric(
    df.loc[df[GROUP_COL].astype(str) == str(GROUP_B), OUTCOME_COL],
    errors="coerce"
).dropna()

# Welch's t-test (does NOT assume equal variances)
t_stat, p_val = stats.ttest_ind(g1, g2, equal_var=False)

print(f"n1={len(g1)} n2={len(g2)}  t={t_stat:.3f}  p={p_val:.4f}")

# Reflection prompts (do not delete)
# 1. What does the null hypothesis assume here?
# 2. Is your OUTCOME_COL truly numeric?
# 3. Does your GROUP_COL contain exactly two groups?
# 4. Interpret the p-value in plain English.

(Note that sample sizes might differ because some penguins are missing some measurements. When we run the t-test, we automatically remove any rows where a value is missing. So the sample size isn’t just “how many males” or “how many females” — it’s how many males and females have valid body mass measurements. In real-world datasets, small differences like this are completely normal.)

### 1.7B‑2 Chi-square template (categorical × categorical)

Example idea (penguins): <code>Species</code> × <code>Island</code> (are species distributed differently across islands?)

Fill in:
- `CAT_COL_A` (categorical)
- `CAT_COL_B` (categorical)

In [ ]:
# TODO: Fill these in (both must be categorical)
CAT_COL_A = None  # e.g., "Species"
CAT_COL_B = None   # e.g., "Island"

assert CAT_COL_A is not None and CAT_COL_B is not None, "Set both CAT_COL_A and CAT_COL_B."

ct = pd.crosstab(df[CAT_COL_A].astype(str), df[CAT_COL_B].astype(str))
display(ct)

chi2, p, dof, expected = stats.chi2_contingency(ct)
print(f"chi2={chi2:.3f} dof={dof} p={p:.4f}")

### 1.7B‑3 ANOVA template (numeric outcome, 3+ groups)

Example idea (penguins): compare mean <code>Body Mass (g)</code> across the 3 species.

Fill in:
- `OUTCOME_COL` (numeric)
- `GROUP_COL` (categorical, with 3+ groups)

In [ ]:
# TODO: Fill these in
OUTCOME_COL = None
GROUP_COL = None

assert OUTCOME_COL is not None and GROUP_COL is not None, "Set OUTCOME_COL and GROUP_COL."

tmp = df[[OUTCOME_COL, GROUP_COL]].copy()
tmp[OUTCOME_COL] = pd.to_numeric(tmp[OUTCOME_COL], errors="coerce")
tmp = tmp.dropna()

# Choose a few largest groups to avoid tiny categories
top_groups = tmp[GROUP_COL].astype(str).value_counts().head(3).index.tolist()
print("Groups used:", top_groups)

groups = [tmp.loc[tmp[GROUP_COL].astype(str) == str(g), OUTCOME_COL].values for g in top_groups]

f_stat, p_val = stats.f_oneway(*groups)
print(f"F={f_stat:.3f}  p={p_val:.4f}")

## 1.7C Advanced — Add a confidence interval (and practical meaning)

**Your task**
1. Choose a t-test style comparison (numeric outcome, 2 groups).  
2. Compute the difference in means (Group A − Group B).  
3. Add a 95% confidence interval for that difference.  
4. Write 2–3 sentences:
   - What does the interval suggest is a plausible range for the effect size?
   - Is that difference *practically* meaningful (not just “p < 0.05”)?

In [ ]:
# TODO: Fill these in (reuse your t-test groups)
OUTCOME_COL = None
GROUP_COL = None

GROUP_A = None
GROUP_B = None

assert OUTCOME_COL is not None and GROUP_COL is not None
assert GROUP_A is not None and GROUP_B is not None

g1 = pd.to_numeric(df.loc[df[GROUP_COL].astype(str) == str(GROUP_A), OUTCOME_COL], errors="coerce").dropna()
g2 = pd.to_numeric(df.loc[df[GROUP_COL].astype(str) == str(GROUP_B), OUTCOME_COL], errors="coerce").dropna()

mean1, mean2 = g1.mean(), g2.mean()
diff = mean1 - mean2

s1, s2 = g1.var(ddof=1), g2.var(ddof=1)
n1, n2 = len(g1), len(g2)

se = np.sqrt(s1/n1 + s2/n2)
df_welch = (s1/n1 + s2/n2)**2 / ((s1**2)/((n1**2)*(n1-1)) + (s2**2)/((n2**2)*(n2-1)))
t_crit = stats.t.ppf(0.975, df_welch)

ci_low = diff - t_crit * se
ci_high = diff + t_crit * se

print("Difference in means (A - B):", diff)
print(f"95% CI: [{ci_low:.2f}, {ci_high:.2f}]")

# 2.7 Hands‑On ML Practice

Your goal: build at least one model, evaluate it, and explain what you learned.

**Important:** Focus on actionable insights, not just metrics.

---

## 2.7A Beginner — Train a logistic regression model + evaluate it

**Your task**
1. Define a binary target column (`TARGET_COL`)  
2. Pick a few feature columns (`FEATURE_COLS`)  
3. Train logistic regression  
4. Report accuracy + confusion matrix

> Hint (penguins): create a binary target from <code>Species</code>, for example:
> “Gentoo vs Adelie” (filter to those two species, then label Gentoo=1, Adelie=0).

In [ ]:
# TODO: Define your target (binary 0/1)
# Suggested penguins pattern:
# 1) df_ml = df.copy()
# 2) df_ml = df_ml[df_ml["Species"].isin([...two species...])].copy()
# 3) df_ml["target"] = (df_ml["Species"] == "...").astype(int)

df_ml = df.copy()

TARGET_COL = None  # e.g., "target"
FEATURE_COLS = []  # e.g., ["Culmen Length (mm)", "Culmen Depth (mm)", "Flipper Length (mm)", "Body Mass (g)", "Island"]

assert TARGET_COL is not None, "Set TARGET_COL to a binary (0/1) column you create."
assert len(FEATURE_COLS) > 0, "Add at least one feature column to FEATURE_COLS."

X = df_ml[FEATURE_COLS].copy()
y = df_ml[TARGET_COL].copy()

# Split (stratify keeps class balance similar in train/test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Separate numeric vs categorical
numeric_features = [c for c in FEATURE_COLS if pd.api.types.is_numeric_dtype(X[c])]
categorical_features = [c for c in FEATURE_COLS if c not in numeric_features]

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]), numeric_features),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features),
    ],
    remainder="drop"
)

log_reg = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=200))
])

log_reg.fit(X_train, y_train)
y_pred = log_reg.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm).plot(values_format="d")
plt.title("Confusion Matrix — Logistic Regression")
plt.show()

## 2.7B Medium — Train a Decision Tree

**Your task**
- Train a decision tree
- Check **accuracy, precision, recall**
- In 2–3 sentences:
   - Is this model performing well?
   - Which metric is lowest?
   - What type of error is the model making most?

In [ ]:
# 1. Create a DecisionTreeClassifier
# 2. Choose a reasonable max_depth
# 3. Plug it into a Pipeline with preprocess
# 4. Compute accuracy, precision, recall using sklearn.metrics

from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

# --- Step 1 & 2: Create the model ---
# TODO: Replace None with an integer (e.g., 3, 4, 5...)
tree_model = DecisionTreeClassifier(
    max_depth=None,   # <-- choose a depth
    random_state=42
)

# --- Step 3: Build the pipeline ---
tree_clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", tree_model)
])

# Fit the model
tree_clf.fit(X_train, y_train)

# Predict
y_pred = tree_clf.predict(X_test)

# --- Step 4: Compute metrics ---
# TODO: Fill in the arguments
acc = accuracy_score(y_test, y_pred)

# If binary classification, this works directly.
# If multiclass, add average="weighted"
prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)

print(f"Accuracy:  {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall:    {rec:.3f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

## 2.7C Advanced — Diagnose overfitting + write one real-world caveat

**Your task**
1. Compare training vs test accuracy for different tree depths  
2. Identify where overfitting begins  
3. Identify the best test depth

In [ ]:
# Overfitting check for decision tree depth (fill the TODOs)
# Assumes: preprocess, X_train, X_test, y_train, y_test, DecisionTreeClassifier, Pipeline, accuracy_score are available.

# Try a range of depths (None means no depth limit)
depths = [2, 4, 8, 16, None]
rows = []

for d in depths:
    clf = Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("model", DecisionTreeClassifier(max_depth=d, random_state=42)),
        ]
    )
    clf.fit(X_train, y_train)

    # TODO: compute train_acc
    # Hint: call clf.predict(X_train) to get predictions on the training set,
    # then pass y_train and those preds into accuracy_score(...)
    # Example hint (don't paste as answer): train_preds = clf.predict(X_train)
    #                                       train_acc = accuracy_score(y_train, train_preds)
    train_acc = None  # <-- replace None with your code

    # TODO: compute test_acc
    # Hint: similar to above but use X_test and y_test
    test_acc = None  # <-- replace None with your code

    rows.append({"max_depth": str(d), "train_acc": train_acc, "test_acc": test_acc})

# Display results
results_df = pd.DataFrame(rows)
results_df

In [ ]:
# Plot train vs test accuracy (run after completing the table above)
# r must be the DataFrame created above (results_df)

r = results_df.copy()
x = np.arange(len(r))

plt.figure(figsize=(8, 4.5))
plt.plot(x, r["train_acc"], marker="o", label="train")
plt.plot(x, r["test_acc"], marker="o", label="test")
plt.xticks(x, r["max_depth"])
plt.ylim(0, 1.05)
plt.xlabel("Decision Tree max_depth")
plt.ylabel("Accuracy")
plt.title("Overfitting check: train vs test accuracy")
plt.legend()
plt.grid(axis="y", alpha=0.25)
plt.show()

# Interpretation hints:
# - Look for the first depth where train accuracy rises but test accuracy stops improving and starts falling.
# - That's the point where overfitting begins.
# - Pick the depth with the highest test accuracy (tie-breaker: prefer smaller depth for simplicity/interpretability).